# Evaluate and Use Models from Model Garden

## Task 2. Prepare prompts for model evaluation

In [1]:
!pip install --upgrade --quiet google-cloud-aiplatform \
    google-cloud-logging litellm 'anthropic[vertex]'

### Setup and Initialize Vertex AI 

Run the following cells to install and import packages and initialize Vertex AI.

In [1]:
import google.cloud.logging
from google.cloud import aiplatform
from IPython.display import Markdown
import logging
import pandas as pd
import random

import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from vertexai.evaluation import (
    MetricPromptTemplateExamples,
    EvalTask,
    PairwiseMetric,
    PointwiseMetric,
)

# Sets Pandas to display content in wider columns
pd.set_option("display.max_colwidth", None)

# Do not remove logging section
client = google.cloud.logging.Client()
client.setup_logging()

# TODO Add your project and location to initialize vertxai
PROJECT_ID, = !gcloud config get project
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

pd.set_option("display.max_colwidth", None)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.appengine_logging_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.appengine_logging_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_pytho

### Define Prompts for Evaluation

In [2]:
prompt_template = """
    You are a technical architect.

    Prepare a high-level project plan for how a solution to the
    following PROJECT REQUEST could be developed using Google Cloud.
    
    We don't want to begin projects that we can't deliver well in
    under four months, so also weigh in on whether you think this
    is a good time for us to attempt this project.
    
    PROJECT REQUEST:
    {project_request}
"""

In [3]:
project_requests = [
    """
    Inventory Management with Real-time Visibility.
    Retail division needs to track inventory across all physical stores
    and e-commerce channels with real-time accuracy. Key pain point
    is stockouts and overstock leading to lost sales and increased
    holding costs. They're looking for a cloud solution that integrates
    seamlessly with existing POS systems and offers predictive analytics
    for demand forecasting.
    """,
    """
    AI-Driven Material Selection & Performance Prediction.
    Product Development is interested in AI's ability to analyze material properties
    and predict their performance under various conditions within the
    context of their industrial designs. Wants assistance in identifying
    the most suitable and cost-effective materials early in the design process.
    """,
    """
    Optimize Yield & Reduce Input Costs.
    Agriculture department specifically mentioned a desire to increase crop
    output (e.g., by X%) while simultaneously decreasing expenses on water,
    fertilizer, and pesticides (e.g., by Y%) through more precise application
    and predictive insights.
    """,
    """
    Enhanced Predictive Maintenance for Fleet Operations.
    Distribution department expressed a strong need to leverage AI/ML on their
    cloud platform for proactive identification of potential component failures
    across their delivery fleet. Key drivers are reducing unscheduled downtime,
    optimizing maintenance schedules, and minimizing operational disruption.
    Requires ingestion of vast amounts of sensor data (IoT), flight logs, and
    maintenance records.
    """,
    """
    Inaccurate Vehicle Valuation.
    Distribution department struggles with inconsistent and often delayed real-time
    pricing for used vehicles, leading to lost profit margins and slower inventory
    turnover. Needs a solution that leverages AI to provide highly accurate,
    dynamic valuation based on a wide range of data points (market trends,
    condition, history, local demand).
    """

]

In [4]:
# TODO
# Update the code so that the variable `prompts` contains a list of
# strings, applying the `prompt_template` to each item in `project_requests`

prompts = [
    prompt_template.format(project_request=project_request)
    for project_request in project_requests
]

print(prompts[0])


    You are a technical architect.

    Prepare a high-level project plan for how a solution to the
    following PROJECT REQUEST could be developed using Google Cloud.
    
    We don't want to begin projects that we can't deliver well in
    under four months, so also weigh in on whether you think this
    is a good time for us to attempt this project.
    
    PROJECT REQUEST:
    
    Inventory Management with Real-time Visibility.
    Retail division needs to track inventory across all physical stores
    and e-commerce channels with real-time accuracy. Key pain point
    is stockouts and overstock leading to lost sales and increased
    holding costs. They're looking for a cloud solution that integrates
    seamlessly with existing POS systems and offers predictive analytics
    for demand forecasting.
    



In [5]:
# Check your work

assert len(prompts) == 5, "You don't have a response for each project_request."
assert type(prompts[0]) == str, ("Make sure your prompts are strings.")

## Task 3. Get Qwen3-Next Instruct Responses using the LiteLLM SDK

In [6]:
import os
import litellm

MAX_TOKENS = 1024

In [7]:
# Vertex AI uses application default credentials
# If not using a Vertex AI resource, use 'gcloud auth application-default login' to authenticate
# Here we unpack from the existing env variable GOOGLE_CLOUD_PROJECT
os.environ["VERTEXAI_PROJECT"] = os.environ["GOOGLE_CLOUD_PROJECT"]
os.environ["VERTEXAI_LOCATION"] = "global"

In [8]:
# Non-streaming response
response = litellm.responses(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-instruct-maas",
    input="Tell me a three sentence bedtime story about a unicorn.",
    max_output_tokens=MAX_TOKENS
)

print(response.output[0].content[0].text)

04:26:06 - LiteLLM:INFO: utils.py:3880 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
04:26:07 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:26:07 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler


Under a velvet sky sprinkled with stars, a tiny silver unicorn named Lumi drifted through the meadow, her horn glowing softly like a whispered wish. She tucked each sleeping animal with a breeze of star-dust—bunnies under clover, owls in their branches—until only the moon remained awake to watch. With a yawn as soft as petals, Lumi curled into a cloud and dreamed of tomorrow’s gentle magic, knowing the world was safe because she believed in it.


In [11]:
import litellm

qwen3_responses = []

for prompt in prompts:
    response = litellm.responses(
        model="vertex_ai/qwen/qwen3-next-80b-a3b-instruct-maas",
        input=prompt,
        max_output_tokens=1024,
    )
    
    # Extract ONLY the text output
    qwen3_responses.append(response.output[0].content[0].text)


04:28:44 - LiteLLM:INFO: utils.py:3880 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
04:28:48 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:48 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:48 - LiteLLM:INFO: utils.py:3880 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
04:28:53 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:53 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:53 - LiteLLM:INFO: utils.py:3880 - 
LiteLLM completion() model= qwen/qwen3-next-80b-a3b-instruct-maas; provider = vertex_ai
04:28:58 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:58 - LiteLLM:INFO: utils.py:1625 - Wrapper: Completed Call, calling success_handler
04:28:58 - LiteLLM:INFO: utils.py:3880 - 
LiteLLM completion() model= 

In [12]:
# Check your work

assert len(prompts) == len(qwen3_responses), "You don't have a response for each prompt."
assert type(qwen3_responses[0]) == str, ("Extract just the text response, "
                                          "not the entire response object.")
# View one response
Markdown(qwen3_responses[0])

Absolutely — as a Technical Architect, I’ll provide a clear, actionable, high-level project plan using **Google Cloud**, along with a go/no-go assessment based on your 4-month delivery constraint.

---

## ✅ **Project Title**:  
**Real-Time Inventory Intelligence Platform for Retail (RTIIP)**

### 🎯 **Objective**:  
Build a cloud-native, real-time inventory visibility and predictive demand forecasting system that integrates with existing POS systems to eliminate stockouts and overstock, reducing lost sales and holding costs by 20–30% within 6 months of launch.

---

## ✅ **Go/No-Go Assessment: Is This a Good Project to Start Now?**

### **✅ YES — Proceed with Confidence**  
**Why?**  
- **Scope is focused and achievable within 4 months** (see plan below).  
- **Google Cloud provides all necessary building blocks** (real-time ingestion, ML, orchestration, APIs) with minimal custom dev.  
- **Retailers universally struggle with this problem** — strong ROI case (reduced stockouts = higher revenue; reduced overstock = lower warehousing costs).  
- **Existing POS systems** are likely already generating structured data — integration is feasible via APIs or batch syncs.  
- **Cloud-native approach** allows rapid iteration and avoids legacy system lock-in.

### ⚠️ **Critical Success Factors**:  
- Must have access to at least **3–6 months of historical sales + inventory data** from POS and warehouse systems.  
- Must secure stakeholder buy-in to **enable API access to POS systems** (critical path).  
- Design for **phased rollout** — pilot 1–2 stores first.

> ✅ **Verdict**: **Proceed** — this is a high-impact, low-to-moderate complexity project that aligns perfectly with Google Cloud’s strengths and your 4-month window.

---

## 🗺️ **High-Level Project Plan (4-Month Timeline)**

### **Phase 1: Discovery & Architecture Finalization (Weeks 1–3)**  
**Goals**: Confirm data sources, define SLAs, finalize architecture.  
**Activities**:  
- Map existing POS systems (e.g., Square, Shopify, Oracle, SAP) — identify API access points or data export formats (CSV/FTP/DB).  
- Audit historical data quality (accuracy, frequency, granularity).  
- Define KPIs:  
  - Stockout rate (%)  
  - Inventory turnover ratio  
  - Forecast accuracy (MAPE)  
- Finalize architecture (see below).  
**Deliverable**: Signed-off architecture diagram + data governance plan.

### **Phase 2: Data Ingestion & Real-Time Pipeline (Weeks 4–7)**  
**Goals**: Build a resilient, real-time inventory data pipeline.  
**Google Cloud Components**:  
- **Cloud Pub/Sub**: Ingest real-time inventory updates from POS and warehouse systems.  
- **Cloud Dataflow (Apache Beam)**: Transform and normalize data (e.g., SKU standardization, store mapping).  
- **BigQuery**: Central data warehouse — store real-time and historical inventory + sales.  
- **Cloud Storage**: Archive raw logs and batch exports.  
**Key Integration**:  
- Use **Cloud Run** or **App Engine** to build lightweight adapters for each POS system (REST/Webhook).  
- Use **Cloud Secret Manager** to securely store API keys.  
**Deliverable**: End-to-end pipeline ingesting 10K+ events/hr with <5s latency.

### **Phase 3: Predictive Demand Forecasting (Weeks 8–11)**  
**Goals**: Deploy ML model to predict demand at SKU × Store level.  
**Google Cloud Components**:  
- **BigQuery ML**: Rapid prototyping of time-series models (e.g., ARIMA, Prophet) on historical sales + promotions + seasonality.  
- **Vertex AI**: For advanced modeling (LSTMs, ensemble models) if BQ ML is insufficient.  
- **Cloud Composer (Airflow)**: Orchestrate daily retraining pipelines (triggered on new sales data).  
**Model Inputs**:  
- Historical sales (daily)  
- Promotions, holidays  
- Store location (urban/rural), foot traffic (if available)  
- Inventory levels (lagged)  
**Output**: Daily SKU-store demand forecast with confidence intervals.  
**Deliverable**: Forecast model deployed, validated on historical data (MAPE < 15%).

### **Phase 4: Real-Time Dashboard & Alerts (Weeks 12–15)**  
**Goals**: Enable visibility and actionability for store managers and planners.  
**Google Cloud Components**:  
- **Looker Studio (formerly Data Studio)**: Interactive dashboard showing:  
  - Real-time inventory levels per store  
  - Forecast vs actual sales  
  - Stockout/overstock alerts  
- **Cloud Functions**:

## Task 4. Get Claude Responses using the Anthropic SDK

In [13]:
from anthropic import AnthropicVertex

In [19]:
CLAUDE_MODEL = "claude-sonnet-4@20250514"
CLAUDE_LOCATIONS = ["us-east5", "europe-west4", "GLOBAL"]

# TODO
# Implement an anthropic_client to access the Claude Sonnet 4 model
# using the model ID defined above
import ipywidgets as widgets
from IPython.display import display

dropdown = widgets.Dropdown(
    options=CLAUDE_LOCATIONS,
    description="Select a location:",
    font_weight="bold",
    style={"description_width": "initial"},
)


def dropdown_eventhandler(change):
    global LOCATION
    if change["type"] == "change" and change["name"] == "value":
        LOCATION = change.new
        print("Selected:", change.new)


LOCATION = dropdown.value
dropdown.observe(dropdown_eventhandler, names="value")
display(dropdown)

anthropic_client = AnthropicVertex(region=LOCATION, project_id=PROJECT_ID)

Dropdown(description='Select a location:', options=('us-east5', 'europe-west4', 'GLOBAL'), style=DescriptionSt…

In [20]:
claude_responses = []

# TODO
# Update the code below to populate the list claude_responses
# with just the text string of Clade Sonnet 4's response to each prompt

for prompt in prompts:
    message = anthropic_client.messages.create(
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model=CLAUDE_MODEL,
    )
    print(message.model_dump_json(indent=2))
    claude_response = message.model_dump_json(indent=2)
    claude_responses.append(claude_response)

{
  "id": "msg_vrtx_018nqEW91LUyerpZ5fJfh4z5",
  "content": [
    {
      "citations": null,
      "text": "# High-Level Project Plan: Real-time Inventory Management System\n\n## Executive Summary\nThis project is **feasible within a 4-month timeline** with proper scope management and phased delivery. The Google Cloud ecosystem provides robust, enterprise-ready services that can deliver immediate value while establishing a foundation for advanced analytics.\n\n## Architecture Overview\n\n### Core Google Cloud Services\n- **Cloud SQL** or **Firestore**: Primary inventory database\n- **Pub/Sub**: Real-time event streaming for inventory updates\n- **Cloud Functions**: Serverless integration and business logic\n- **BigQuery**: Data warehouse for analytics and forecasting\n- **Vertex AI**: Machine learning for demand prediction\n- **Cloud Run**: Microservices for API layer\n- **API Gateway**: Secure API management\n- **Cloud Monitoring**: Observability and alerting\n\n## 4-Month Phased Deli

In [21]:
# Check your work

assert len(prompts) == len(claude_responses), "You don't have a response for each prompt."
assert type(claude_responses[0]) == str, ("Extract just the text response, "
                                          "not the entire response object.")

# View one response
Markdown(claude_responses[0])

{
  "id": "msg_vrtx_018nqEW91LUyerpZ5fJfh4z5",
  "content": [
    {
      "citations": null,
      "text": "# High-Level Project Plan: Real-time Inventory Management System\n\n## Executive Summary\nThis project is **feasible within a 4-month timeline** with proper scope management and phased delivery. The Google Cloud ecosystem provides robust, enterprise-ready services that can deliver immediate value while establishing a foundation for advanced analytics.\n\n## Architecture Overview\n\n### Core Google Cloud Services\n- **Cloud SQL** or **Firestore**: Primary inventory database\n- **Pub/Sub**: Real-time event streaming for inventory updates\n- **Cloud Functions**: Serverless integration and business logic\n- **BigQuery**: Data warehouse for analytics and forecasting\n- **Vertex AI**: Machine learning for demand prediction\n- **Cloud Run**: Microservices for API layer\n- **API Gateway**: Secure API management\n- **Cloud Monitoring**: Observability and alerting\n\n## 4-Month Phased Delivery Plan\n\n### Phase 1: Foundation (Month 1)\n**Deliverables:**\n- Core inventory database design and deployment\n- Basic API framework for CRUD operations\n- POS system integration framework\n- Real-time inventory tracking for 2-3 pilot stores\n\n**Key Activities:**\n- Requirements gathering and data mapping\n- Set up Cloud SQL instance with inventory schema\n- Implement Cloud Functions for basic inventory operations\n- Establish Pub/Sub topics for real-time updates\n- Build initial integration with primary POS system\n\n### Phase 2: Integration & Real-time Processing (Month 2)\n**Deliverables:**\n- Complete POS integration across all store formats\n- E-commerce platform integration\n- Real-time inventory synchronization\n- Basic dashboard for inventory visibility\n\n**Key Activities:**\n- Implement remaining POS integrations using Cloud Functions\n- Set up real-time data pipelines with Pub/Sub\n- Deploy Cloud Run services for API layer\n- Build inventory management dashboard\n- Implement stock level alerts and notifications\n\n### Phase 3: Analytics & Intelligence (Month 3)\n**Deliverables:**\n- Historical data migration to BigQuery\n- Basic demand forecasting model\n- Inventory optimization recommendations\n- Advanced reporting and analytics\n\n**Key Activities:**\n- Data pipeline from operational systems to BigQuery\n- Develop initial ML models using Vertex AI\n- Implement predictive analytics for top-selling items\n- Create executive dashboards and reports\n- Set up automated reorder point calculations\n\n### Phase 4: Optimization & Launch (Month 4)\n**Deliverables:**\n- Production-ready system with full monitoring\n- Advanced predictive capabilities\n- Performance optimization\n- User training and documentation\n\n**Key Activities:**\n- Performance tuning and optimization\n- Enhanced ML models for seasonal trends\n- Comprehensive testing and quality assurance\n- User acceptance testing and training\n- Production deployment with monitoring\n\n## Technical Implementation Strategy\n\n### Integration Approach\n1. **API-First Design**: RESTful APIs using Cloud Run for scalability\n2. **Event-Driven Architecture**: Pub/Sub for real-time inventory updates\n3. **Microservices**: Separate services for inventory, forecasting, and reporting\n4. **Data Lake Strategy**: Raw data in Cloud Storage, processed data in BigQuery\n\n### Key Integrations\n- **POS Systems**: Cloud Functions with appropriate connectors\n- **E-commerce Platforms**: API integrations via Cloud Run services\n- **ERP Systems**: Batch and real-time sync capabilities\n- **External Data**: Weather, events, and market data for enhanced forecasting\n\n## Risk Assessment & Mitigation\n\n### High Risks\n1. **POS Integration Complexity**\n   - *Mitigation*: Start with pilot stores, use established connectors where possible\n2. **Data Quality Issues**\n   - *Mitigation*: Implement data validation and cleansing in Cloud Functions\n3. **Real-time Performance Requirements**\n   - *Mitigation*: Use Pub/Sub for async processing, implement caching strategies\n\n### Medium Risks\n1. **ML Model Accuracy**\n   - *Mitigation*: Start with simple models, iterate based on feedback\n2. **User Adoption**\n   - *Mitigation*: Involve key users in design, provide comprehensive training\n\n## Success Metrics\n- **Inventory Accuracy**: >95% real-time accuracy\n- **Stockout Reduction**: 30% decrease in stockouts\n- **Forecast Accuracy**: 80% accuracy for 30-day demand prediction\n- **System Performance**: <2 second response times for inventory queries\n\n## Resource Requirements\n- **Technical Team**: 1 Cloud Architect, 2 Full-stack Developers,",
      "type": "text"
    }
  ],
  "model": "claude-sonnet-4-20250514",
  "role": "assistant",
  "stop_reason": "max_tokens",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "input_tokens": 193,
    "output_tokens": 1024,
    "server_tool_use": null,
    "service_tier": null
  }
}

## Task 5. Conduct a pairwise evaluation

### Explore Metrics and Select Evaluation Criteria

In [22]:
MetricPromptTemplateExamples.list_example_metric_names()

['coherence',
 'fluency',
 'safety',
 'groundedness',
 'instruction_following',
 'verbosity',
 'text_quality',
 'summarization_quality',
 'question_answering_quality',
 'multi_turn_chat_quality',
 'multi_turn_safety',
 'pairwise_coherence',
 'pairwise_fluency',
 'pairwise_safety',
 'pairwise_groundedness',
 'pairwise_instruction_following',
 'pairwise_verbosity',
 'pairwise_text_quality',
 'pairwise_summarization_quality',
 'pairwise_question_answering_quality',
 'pairwise_multi_turn_chat_quality',
 'pairwise_multi_turn_safety']

In [23]:
print(MetricPromptTemplateExamples.Pairwise.TEXT_QUALITY.metric_prompt_template)


# Instruction
You are an expert evaluator. Your task is to evaluate the quality of the responses generated by two AI models. We will provide you with the user input and a pair of AI-generated responses (Response A and Response B). You should first read the user input carefully for analyzing the task, and then evaluate the quality of the responses based on the Criteria provided in the Evaluation section below.
You will first judge responses individually, following the Rating Rubric and Evaluation Steps. Then you will give step-by-step explanations for your judgment, compare the results to declare the winner based on the Rating Rubric and Evaluation Steps.

# Evaluation
## Metric Definition
You will be assessing the Text Quality of each model's response, which measures how effectively the text conveys clear, accurate, and engaging information that directly addresses the user's prompt, considering factors like fluency, coherence, relevance, and conciseness.

## Criteria
Coherence: The re

### Prepare and Execute the Evaluation Task

In [26]:
# TODO
# Provide all the variables required for the metric
# selected above

eval_dataset=pd.DataFrame(
    {
        "prompt": ["prompt0", "prompt1"],
    }
)
eval_dataset = pd.DataFrame(
    {
        "prompt": prompts,
        "baseline_model_response": qwen3_responses,
        "response": claude_responses,
    }
)

eval_dataset.head(1)

prompt  \
0  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Inventory Management with Real-time Visibility.\n    Retail division needs to track inventory across all physical stores\n    and e-commerce channels with real-time accuracy. Key pain point\n    is stockouts and overstock leading to lost sales and increased\n    holding costs. They're looking for a cloud solution that integrates\n    seamlessly with existing POS systems and offers predictive analytics\n    for demand forecasting.\n    \n   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [27]:
selected_metric = MetricPromptTemplateExamples.Pairwise.TEXT_QUALITY

# TODO
# Complete the definition of the EvalTask using the dataset defined
# above and the selected metric. Keep the provided experiment name.

# pairwise_eval_task = EvalTask(
#     experiment="pairwise-text-quality-001",
# )
pairwise_eval_task = EvalTask(
    dataset=eval_dataset,
    metrics=[selected_metric],
    experiment="pairwise-text-quality-001",
)


In [28]:
pairwise_result = pairwise_eval_task.evaluate()

Associating projects/5399527077/locations/us-central1/metadataStores/default/contexts/pairwise-text-quality-001-52a31f59-36c4-42e0-9ebb-72da2657e5f8 to Experiment: pairwise-text-quality-001


Computing metrics with a total of 5 Vertex Gen AI Evaluation Service API requests.


100%|██████████| 5/5 [00:21<00:00,  4.36s/it]

All 5 metric requests are successfully computed.
Evaluation Took:21.908645441999397 seconds


In [29]:
# TODO
# Display the result's summary_metrics
pairwise_result.summary_metrics

{'row_count': 5,
 'pairwise_text_quality/candidate_model_win_rate': np.float64(0.0),
 'pairwise_text_quality/baseline_model_win_rate': np.float64(1.0)}

In [30]:
# TODO
# Display the result's metrics_table table
pairwise_result.metrics_table

prompt  \
0                                               \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Inventory Management with Real-time Visibility.\n    Retail division needs to track inventory across all physical stores\n    and e-commerce channels with real-time accuracy. Key pain point\n    is stockouts and overstock leading to lost sales and increased\n    holding costs. They're looking for a cloud solution that integrates\n    seamlessly with existing POS systems and offers predictive analytics\n    for demand forecasting.\n    \n   
1                                                                                                                  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    AI-Driven Material Selection & Performance Prediction.\n    Product Development is interested in AI's ability to analyze material properties\n    and predict their performance under various conditions within the\n    context of their industrial designs. Wants assistance in identifying\n    the most suitable and cost-effective materials early in the design process.\n    \n   
2                                                                                                                                                                                     \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Optimize Yield & Reduce Input Costs.\n    Agriculture department specifically mentioned a desire to increase crop\n    output (e.g., by X%) while simultaneously decreasing expenses on water,\n    fertilizer, and pesticides (e.g., by Y%) through more precise application\n    and predictive insights.\n    \n   
3  \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think this\n    is a good time for us to attempt this project.\n    \n    PROJECT REQUEST:\n    \n    Enhanced Predictive Maintenance for Fleet Operations.\n    Distribution department expressed a strong need to leverage AI/ML on their\n    cloud platform for proactive identification of potential component failures\n    across their delivery fleet. Key drivers are reducing unscheduled downtime,\n    optimizing maintenance schedules, and minimizing operational disruption.\n    Requires ingestion of vast amounts of sensor data (IoT), flight logs, and\n    maintenance records.\n    \n   
4                                                                                        \n    You are a technical architect.\n\n    Prepare a high-level project plan for how a solution to the\n    following PROJECT REQUEST could be developed using Google Cloud.\n    \n    We don't want to begin projects that we can't deliver well in\n    under four months, so also weigh in on whether you think 

In [31]:
# Explore the first result's explanation
print(pairwise_result.metrics_table.loc[0,'pairwise_text_quality/explanation'])

BASELINE response provides a more detailed and specific project plan, particularly in its integration of Google Cloud services within each phase, which directly addresses the 'how a solution could be developed' instruction. Its 'Go/No-Go Assessment' is also more comprehensive and well-justified than CANDIDATE response's. Both responses unfortunately cut off abruptly due to token limits, but BASELINE response's content prior to the cut-off is marginally stronger in addressing the core aspects of the prompt.


## Test the CodeGemma deployment

In [32]:
import google.cloud.logging
import logging

from google.cloud import aiplatform

# Do not remove logging section
client = google.cloud.logging.Client()
client.setup_logging()


# Replace your project number and endpoint id

endpoint_resource_name=f"projects/PROJECT_NUMBER/locations/us-central1/endpoints/ENDPOINT_ID"
endpoint=aiplatform.Endpoint(endpoint_resource_name)


In [33]:

prompt = "Write a function to list n Fibonacci numbers in Python." 
max_tokens = 500 
temperature = 1.0 
top_p = 1.0  
top_k = 1 
raw_response = True

# instances = [
#     {
#      # Fill in the appropriate configuration
#     },
# ]
instances = [
    {
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "raw_response": raw_response,
    }
]


response = endpoint.predict(
    instances=instances
)

# "<|file_separator|>" is the end of the file token.
for prediction in response.predictions:
    print(prediction.split("<|file_separator|>")[0])


print(response.deployed_model_id)

# Do not remove logging section
log_message = f"Fibonacci function: {response}"
logging.info(log_message)

py
<|fim_prefix|><|fim_suffix|><|fim_middle|>def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    else:
        fib_sequence = [0, 1]
        while len(fib_sequence) < n:
            next_number = fib_sequence[-1] + fib_sequence[-2]
            fib_sequence.append(next_number)
        return fib_sequence

n = int(input("Enter the number of Fibonacci numbers to generate: "))
fibonacci_numbers = fibonacci(n)
print(fibonacci_numbers)

4248548663854366720


## Perform Cleanup

In [34]:
# Delete all endpoints and their deployed models
endpoints = aiplatform.Endpoint.list()

if not endpoints:
    print("No endpoints found in this project/region.")
else:
    for endpoint in endpoints:
        print(f"\nEndpoint: {endpoint.display_name} ({endpoint.resource_name})")

        deployed_models = endpoint.list_models()
        for dm in deployed_models:
            print(f"  Undeploying model ID: {dm.id}")            
            # TODO Uncomment the statement below to undeploy a model
            endpoint.undeploy(deployed_model_id=dm.id, sync=True)
            print(f"  Undeployed model ID: {dm.id}")

        print("  Deleting endpoint...")
        endpoint.delete(sync=True)
        print(f"  Deleted endpoint: {endpoint.display_name}")

print("\nDeleting models...")
models = aiplatform.Model.list()

for model in models:
    print(f"  Deleting model: {model.display_name}")
    # TODO Uncomment the statement below to delete the model.
    model.delete(sync=True)
    print("  Deleted.")
    
# Do not remove logging section
log_message = "Deleted endpoints"
logging.info(log_message)



Endpoint: google_codegemma-2b-mg-one-click-deploy (projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b)
  Undeploying model ID: 4248548663854366720
Undeploying Endpoint model: projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b
Undeploy Endpoint model backing LRO: projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b/operations/2412620964253663232
Endpoint model undeployed. Resource name: projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b
  Undeployed model ID: 4248548663854366720
  Deleting endpoint...
Deleting Endpoint : projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b
Endpoint deleted. . Resource name: projects/5399527077/locations/us-central1/endpoints/mg-endpoint-496aac9f-f7f6-42aa-83c2-b2f9ff9b445b
Deleting Endpoint resource: projects/5